# Free Qwen catalog selector

128 authored synthetic training requests and 48 development requests. Select one of 32 existing drafts or return unknown. Fish is excluded. No images, geometry targets or physical-success labels are added.

Use a free GPU runtime. Run setup, approve the usual Drive connection if requested, then run the pilot. Base evaluation, two-epoch QLoRA and adapter evaluation save to Drive. No paid services or quota workarounds. A successful selection score does not establish buildability; the adapter is never automatically deployed. The previous unsuccessful geometry experiment is preserved separately.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError("No eligible free CUDA GPU. Select a free GPU runtime or stop and retry later; do not upgrade.")
from google.colab import drive
drive.mount("/content/drive")
repo=Path("/content/beadform-catalog-selector-v1")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","runpod-training-guide","https://github.com/c-tustin/beadform.git",str(repo)],check=True)
run=Path("/content/drive/MyDrive/Beadform-free-qwen/runs/catalog-selector-v1")
run.mkdir(parents=True,exist_ok=True)
revision=subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()
source_file=run/"repository-revision.txt"
if source_file.exists() and source_file.read_text().strip()!=revision:
    raise RuntimeError("Saved run uses a different repository revision. Preserve its files and restore that revision.")
source_file.write_text(revision+"\n")
print("Repository",revision,"GPU",torch.cuda.get_device_name(0),flush=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo/"training/requirements.txt")],check=True)
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
freeze_file=run/"pip-freeze.txt"
if freeze_file.exists() and freeze_file.read_text()!=freeze:
    raise RuntimeError("Saved environment differs. Restore recorded packages before resuming.")
freeze_file.write_text(freeze)
subprocess.run([sys.executable,str(repo/"training/train_catalog_selector.py"),"--output",str(run),"--check"],cwd=repo,check=True)


In [ ]:
command=[sys.executable,"-u",str(repo/"training/train_catalog_selector.py"),"--output",str(run),"--epochs","2","--resume"]
if not (run/"comparison.json").exists():
    with (run/"run.log").open("a") as log:
        process=subprocess.Popen(command,cwd=repo,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line);log.flush();print(line,end="",flush=True)
        code=process.wait()
    if code:
        raise RuntimeError(f"Pilot stopped with code {code}. Inspect the saved log; no adapter was deployed.")
print((run/"comparison.json").read_text())
print("Saved:",run)
print("Development selection only; inspect rejections and independent requests. No automatic deployment.")
